# Lab 1: Computer Vision - Bird's Eye View (BEV) and Noise Filtering

Welcome to Lab 1. In this lab, we will learn how to process camera images to estimate the real-world distance of an object.

---

## Overview

When developing perception systems for autonomous vehicles, wide-angle cameras suffer from **perspective distortion** — distant objects appear smaller and parallel lines converge toward a vanishing point. This makes direct pixel-based distance estimation unreliable.

To address this issue, we apply **Inverse Perspective Mapping (IPM)** to transform the image into a **Bird’s Eye View (BEV)**. This removes perspective distortion and creates a linear space suitable for distance measurement.

Finally, the computed distance is smoothed using an **Exponential Moving Average (EMA)** filter to reduce noise and prevent unstable control behavior.

---

## Learning Objectives

After completing this lab, you will be able to:

- Construct a perspective transformation matrix (Homography)
- Transform coordinates from image space to BEV space
- Calibrate pixel-to-centimeter conversion
- Implement and tune an EMA filter

---

## Problem Statement

**Goal**: Estimate the real-world distance from the camera to an object.

### Requirements

- **Input**: Bounding Box `(x1, y1, x2, y2)`
- **Output**: Smoothed real-world distance (cm)

In [ ]:
import cv2
import numpy as np

# 1. Inverse Perspective Mapping (IPM)

## Theory

IPM uses a **Homography Matrix (3x3)** to map points from the camera view into a top-down BEV space.

Transformation:

[x', y', w'] = H * [u, v, 1]

Normalization:

x = x' / w'  
y = y' / w'

The matrix is computed by mapping:
- 4 source points (trapezoid in image)
- 4 destination points (rectangle in BEV)

In [ ]:
def create_perspective_matrix(width, height):
    src_ratios = ((0.35, 0.45), (0.25, 0.90), (0.75, 0.90), (0.65, 0.45))
    dst_ratios = ((0.25, 0.00), (0.25, 1.00), (0.75, 1.00), (0.75, 0.00))

    src = np.float32([[...]  for x, y in src_ratios])
    dst = np.float32([[...]  for x, y in dst_ratios])

    M = cv2.getPerspectiveTransform(src, dst)
    return M

## Transforming Object Coordinates

We use the **bottom-center point** of the bounding box to represent the object's position on the ground.

In [ ]:
def transform_point(cx, cy, M):
    point = np.array([[[float(cx), float(cy)]]], dtype=np.float32)
    
    point_bev = [...] 
    
    return point_bev[0][0]

# 2. Distance Estimation (Calibration)

## Theory

After BEV transformation, the vertical axis represents distance.

Distance formula:

distance = distance_pixel * cm_per_pixel

Where:
- distance_pixel = distance from bottom of frame to object
- cm_per_pixel = calibration factor

In [ ]:
def calculate_real_distance(y_bev, frame_height):
    
    distance_pixel = [...] 

    if distance_pixel <= 0:
        return 9999.0

    cm_per_pixel = [...] 
    return distance_pixel * cm_per_pixel

# 3. Exponential Moving Average (EMA)

EMA smooths noisy signals:

S_t = alpha * X_t + (1 - alpha) * S_(t-1)

- alpha small → smoother but slower
- alpha large → faster but noisier

In [ ]:
def apply_ema_filter(raw, prev, alpha=0.4):
    if prev is None:
        return raw
    
    smoothed = [...] 
    return smoothed

# 4. Full Pipeline Test

In [ ]:
# Giả lập bounding box
x1, y1, x2, y2 = 100, 120, 200, 300

# Tính bottom-center
cx = (x1 + x2) / 2
cy = y2

# Tạo ma trận
M = create_perspective_matrix(640, 480)

# Transform
x_bev, y_bev = transform_point(cx, cy, M)

# Distance
raw_dist = calculate_real_distance(y_bev, 480)

# EMA
filtered = apply_ema_filter(raw_dist, None)

print("Raw distance:", raw_dist)
print("Filtered distance:", filtered)